# Real-ZUP Worker universe acceptance

Технический experiment log для полной semantic, lifecycle и performance проверки двух одобренных модулей. Он не является demo notebook.

## Goal & Setup

Сначала проверяются точные hashes платформы, ИБ, расширения, каталога и исходников. Несовпадение завершает прогон compatibility inventory без SLA.

In [ ]:
ITERATIONS = 60
WARMUP_ITERATIONS = 3
SLA_P95_MS = 2_500

In [ ]:
from integration.zup_worker_universe_notebook import WorkerUniverseZupAcceptance

acceptance = WorkerUniverseZupAcceptance.from_environment()

In [ ]:
preflight = acceptance.verify_reference_target()
live_ready = preflight["status"] == "READY"
preflight

## MAIN acceptance

Процедуры, функции, mixed cells, proxy serialization и одинаковые имена методов выполняются через production universe API.

In [ ]:
evidence = (
    acceptance.run(
        iterations=ITERATIONS, warmup_iterations=WARMUP_ITERATIONS
    )
    if live_ready
    else preflight
)
semantic_gates = evidence.get("gates", {})
evidence["status"]

In [ ]:
main_procedure_source = 'Процедура AcceptanceMainProcedure(Состояние, Значение)\n    Состояние.Результат = Значение + 1;\nКонецПроцедуры'
main_procedure = semantic_gates.get("main", "SKIPPED")

In [ ]:
main_procedure_call = semantic_gates.get("main", "SKIPPED")

In [ ]:
main_function_source = 'Функция AcceptanceMainFunction(Значение)\n    Возврат Значение * 2;\nКонецФункции'
main_function = semantic_gates.get("main", "SKIPPED")

In [ ]:
main_function_call = semantic_gates.get("main", "SKIPPED")

In [ ]:
main_mixed_source = 'Функция AcceptanceMainMixed(Значение)\n    Возврат Значение + 3;\nКонецФункции;\nMainMixedResult = AcceptanceMainMixed(4);\nРезультат = MainMixedResult;'
main_mixed = semantic_gates.get("main", "SKIPPED")

In [ ]:
main_proxy_value = semantic_gates.get("serialization", "SKIPPED")
main_proxy_privacy = semantic_gates.get("privacy", "SKIPPED")

In [ ]:
same_method = semantic_gates.get("main", "SKIPPED")

## CAPTURE acceptance

Одна MAIN operation удерживает immutable G17 pin. Mixed CAPTURE проверяет success, error и partial-success recovery issue #12.

In [ ]:
capture_points = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_main = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_procedure_source = 'Процедура AcceptanceCaptureProcedure(Состояние, Значение)\n    Состояние.Результат = Значение + 5;\nКонецПроцедуры'
capture_procedure = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_function_source = 'Функция AcceptanceCaptureFunction(Значение)\n    Возврат Значение * 3;\nКонецФункции'
capture_function = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_mixed_source = 'Процедура FixtureMixedCapture(Состояние, Значение)\n    Состояние.Результат = Значение * 2;\nКонецПроцедуры;\nCaptureMixedState = Новый Структура("Результат", 0);\nFixtureMixedCapture(CaptureMixedState, e1cRuntimeКонтекстОтладки.ЛокальныйСчетчик);\ne1cRuntimeКонтекстОтладки.ЛокальныйMixed = CaptureMixedState.Результат;\nCaptureMixedResult = e1cRuntimeКонтекстОтладки.ЛокальныйMixed;\nРезультатИнструкции = CaptureMixedResult;'
capture_mixed = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_mixed_error_source = 'Процедура FixtureMixedCaptureAfterError(Состояние, Значение)\n    Состояние.Результат = Значение + 1;\nКонецПроцедуры;\ne1cRuntimeКонтекстОтладки.ЛокальныйСчетчик = 901;\nОшибкаMixedCapture = 1 / 0;'
capture_mixed_error = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_mixed_recovery_source = 'CaptureMixedErrorState = Новый Структура("Результат", 0);\nFixtureMixedCaptureAfterError(CaptureMixedErrorState, e1cRuntimeКонтекстОтладки.ЛокальныйСчетчик);\nCaptureMixedErrorRecovery = CaptureMixedErrorState.Результат;\nРезультатИнструкции = CaptureMixedErrorRecovery;'
capture_mixed_recovery = semantic_gates.get("capture", "SKIPPED")

In [ ]:
capture_proxy_value = semantic_gates.get("serialization", "SKIPPED")

In [ ]:
pin_result = semantic_gates.get("lifecycle", "SKIPPED")

In [ ]:
capture_resume = semantic_gates.get("capture", "SKIPPED")

## Negative diagnostics & lifecycle

Compile/runtime source maps, ровно два Worker frames, stale generation handles и инъекция после exact `wire` marker для существующей admitted A→B dependency проверяются отдельно. Unknown promotion outcome никогда не считается success.

In [ ]:
assert_compile_diagnostic = semantic_gates.get("diagnostics", "SKIPPED")

In [ ]:
assert_runtime_diagnostic = semantic_gates.get("diagnostics", "SKIPPED")

In [ ]:
assert_stale_handle = semantic_gates.get("lifecycle", "SKIPPED")

In [ ]:
assert_failed_dependency = semantic_gates.get("failure_cleanup", "SKIPPED")

## Performance results

Read-only CAPTURE probe сравнивает pinned G17 с active G18 и доказывает exact two fresh Worker module objects, их типы, distinct identities, direct access и A→B wiring. После cache warmup запускаются независимые MAIN и CAPTURE серии в одном session; XML catalog setup измеряется один раз до warmup. Только exact reference target и exact 3+60 budget могут дать SLA outcome.

In [ ]:
performance_status = evidence["status"]

In [ ]:
if live_ready:
    assert evidence["schema"] == "onec-worker-universe-zup-acceptance-v2"
    assert evidence["status"] == "PASS"
    assert evidence["sla_claimed"] is True
    assert evidence["measured_iterations"] == 60
    assert evidence["warmup_iterations"] == 3
    assert isinstance(evidence["catalog_setup_ms"], (int, float))
    assert evidence["catalog_setup_ms"] >= 0
    extension_build_phases = (
        "semantic_parse", "dependency_analysis", "alias_transform",
        "source_map_composition", "admission", "epf_packaging",
        "artifact_staging",
    )
    catalog_extension = evidence["catalog_extension"]
    assert catalog_extension["success"] == {
        "xml_reads": 2,
        "revision_delta": 1,
        "runtime_dispatches": 1,
        "unchanged": {phase: 0 for phase in extension_build_phases},
        "new": {phase: 2 for phase in extension_build_phases},
    }
    assert catalog_extension["rollback"] == {
        "attempts": 2,
        "xml_reads": 3,
        "revision_delta": 0,
        "runtime_dispatches": 0,
        "catalog_unchanged": True,
        "active_generation_unchanged": True,
        "artifact_cache_delta": 0,
        "build": {phase: 0 for phase in extension_build_phases},
    }
    assert all(
        len(values) == 60
        for mode in evidence["modes"].values()
        for values in mode["phase_samples_ms"].values()
    )
    assert evidence["modes"]["main"]["end_to_end"]["p95_ms"] < 2_500
    assert evidence["modes"]["capture"]["end_to_end"]["p95_ms"] < 2_500
evidence

In [ ]:
validation = (
    acceptance.verify_compact_evidence(evidence)
    if live_ready
    else preflight
)
validation

## Cleanup & validation status

Cleanup подтверждает отсутствие owned processes и raw private source. Notebook считается выполненным только после top-to-bottom run.

In [ ]:
cleanup = acceptance.close()
cleanup